# Multimodal Agent Pattern
### When the process uses more than text

Input data for some business processes arrive as PDFs, images, charts, and mixed-format documents. A multimodal agent **perceives** these inputs, extracts structured data, and reasons over what it finds.

This implementation uses **Amazon Nova 2 Lite** — a lightweight multimodal model that can analyze large document sets at optimized cost.

### The JSON Schema Pattern

We define a JSON schema specifying the exact fields the model should extract. The schema is embedded in the prompt — the model reads the PDF and returns a conforming JSON object.

### Business Use Case: AnyComp Telecom Network Performance Report Analysis

Instead of a human analyst manually reviewing monthly reports, an agent extracts key metrics, identifies issues, and routes findings to the right teams.

### Architecture

```
PDF Report → Agent → extract_from_pdf()
                         │
                    Read PDF bytes
                         │
                    Embed JSON schema in prompt
                         │
                    bedrock_runtime.converse(Nova 2 Lite)
                         │
                    Structured JSON output
                         │
                    Agent reasons over extracted data
                         │
                    Executive summary + issues + team routing
```



## Install Dependencies

Run this cell first, then **restart the kernel** and continue.

In [ ]:
!pip install -q strands-agents>=1.34.0 strands-agents-tools boto3 "typing_extensions>=4.12.0"

## Setup

In [ ]:
import boto3
import json
import re
from pathlib import Path
from typing import List

REGION = 'us-east-1'
MODEL_ID = 'us.amazon.nova-2-lite-v1:0'

bedrock_runtime = boto3.client('bedrock-runtime', region_name=REGION)

# Path to the network performance report
# Find the report — handle different kernel working directories
REPORT_PATH = None
for candidate in [
    Path('../common/sample_data/Anycom Network Performance_March2026.pdf'),
    Path('common/sample_data/Anycom Network Performance_March2026.pdf'),
    Path('../../agent-orchestration-patterns/common/sample_data/Anycom Network Performance_March2026.pdf'),
]:
    if candidate.exists():
        REPORT_PATH = candidate
        break
assert REPORT_PATH, 'Network Performance report not found. Check common/sample_data/'
assert REPORT_PATH.exists(), f'Report not found: {REPORT_PATH}'
print(f'Report: {REPORT_PATH.name} ({REPORT_PATH.stat().st_size:,} bytes)')

## Preview the Document

This is the PDF the agent will analyze. In a real pipeline, this arrives from S3 or a document management system.

In [ ]:
from IPython.display import IFrame, display
display(IFrame(str(REPORT_PATH), width=800, height=600))

## Define the Extraction Schema

The schema defines the exact fields Nova should extract from the network performance report. It’s embedded directly in the prompt — Nova reads the PDF and returns a JSON object conforming to this structure.

In [ ]:
NETWORK_REPORT_SCHEMA = {
    "type": "object",
    "description": "Structured data extracted from a monthly network performance report",
    "properties": {
        "report_metadata": {
            "type": "object",
            "description": "Report identification and period",
            "properties": {
                "report_title": {"type": "string"},
                "reporting_period": {"type": "string"},
                "prepared_by": {"type": "string"},
                "date_generated": {"type": "string", "format": "date"}
            }
        },
        "network_summary": {
            "type": "object",
            "description": "High-level network health metrics",
            "properties": {
                "overall_uptime_pct": {"type": "number"},
                "total_incidents": {"type": "integer"},
                "critical_incidents": {"type": "integer"},
                "avg_latency_ms": {"type": "number"},
                "avg_packet_loss_pct": {"type": "number"},
                "total_subscribers": {"type": "integer"}
            }
        },
        "regional_performance": {
            "type": "array",
            "description": "Performance metrics broken down by region",
            "items": {
                "type": "object",
                "properties": {
                    "region": {"type": "string"},
                    "uptime_pct": {"type": "number"},
                    "incidents": {"type": "integer"},
                    "avg_latency_ms": {"type": "number"},
                    "subscriber_count": {"type": "integer"},
                    "status": {"type": "string"}
                }
            }
        },
        "top_issues": {
            "type": "array",
            "description": "Most significant issues identified in the report",
            "items": {
                "type": "object",
                "properties": {
                    "issue": {"type": "string"},
                    "severity": {"type": "string"},
                    "affected_region": {"type": "string"},
                    "impact": {"type": "string"},
                    "recommended_action": {"type": "string"}
                }
            }
        },
        "recommendations": {
            "type": "array",
            "description": "Action items and recommendations from the report",
            "items": {"type": "string"}
        }
    },
    "required": ["report_metadata", "network_summary", "regional_performance", "top_issues"]
}

print('Schema defined with sections:')
for key in NETWORK_REPORT_SCHEMA['properties']:
    print(f'   {key}')

## Core Extraction Function

This helper reads the PDF bytes, embeds the schema in the prompt, and calls Nova 2 Lite via `bedrock_runtime.converse()`. All three framework implementations use this same function.

In [ ]:
def extract_from_pdf(file_path: str, schema: dict) -> dict:
    """Extract structured data from a PDF using Nova 2 Lite with schema-in-prompt.

    Reads raw PDF bytes, embeds the JSON schema in the prompt, and calls
    Nova via bedrock_runtime.converse. Returns parsed JSON.
    """
    with open(file_path, 'rb') as f:
        pdf_bytes = f.read()

    schema_str = json.dumps(schema.get('properties', schema), indent=2)

    prompt = (
        "You are a document extraction specialist processing a Network Performance Report.\n"
        "Extract all information from this document and return ONLY a valid JSON object "
        "with the following fields:\n"
        f"{schema_str}\n"
        "Rules:\n"
        "- Use null for any field you cannot find in the document\n"
        "- Do not guess or hallucinate values\n"
        "- Return dates in YYYY-MM-DD format\n"
        "- For arrays, include all items found in the document\n"
        "- Return ONLY the JSON object — no explanation, no markdown fences, no extra text"
    )

    response = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{
            "role": "user",
            "content": [
                {
                    "document": {
                        "format": "pdf",
                        "name": "network_report",
                        "source": {"bytes": pdf_bytes},
                    }
                },
                {"text": prompt},
            ],
        }],
        additionalModelRequestFields={
            "reasoningConfig": {"type": "enabled", "maxReasoningEffort": "medium"}
        },
        inferenceConfig={"maxTokens": 8096},
    )

    # Parse response
    output_text = ""
    for block in response.get("output", {}).get("message", {}).get("content", []):
        if isinstance(block, dict) and block.get('text', '').strip():
            output_text = block['text'].strip()
            break

    # Clean markdown fences if present
    if output_text.startswith("```"):
        lines = output_text.splitlines()
        output_text = "\n".join(l for l in lines if not l.strip().startswith("```")).strip()

    try:
        return json.loads(output_text)
    except json.JSONDecodeError:
        json_match = re.search(r'\{[\s\S]+\}', output_text)
        if json_match:
            try:
                return json.loads(json_match.group())
            except json.JSONDecodeError:
                pass
    return {"error": "JSON parse failed", "raw_response": output_text[:500]}


print("extract_from_pdf() defined")

## Implementation 1: Strands Agents

The Strands agent has a `@tool` that calls `extract_from_pdf`, then reasons over the extracted data to produce a summary and route issues to the right teams.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def analyze_network_report(report_path: str) -> str:
    """Extract and analyze a network performance report PDF.

    Args:
        report_path: Path to the PDF report file
    """
    extracted = extract_from_pdf(report_path, NETWORK_REPORT_SCHEMA)
    return json.dumps(extracted, indent=2)


strands_model = BedrockModel(
    model_id="us.amazon.nova-2-lite-v1:0",
    region_name=REGION,
    max_tokens=8096,
    additional_request_fields={
        "reasoningConfig": {"type": "enabled", "maxReasoningEffort": "medium"}
    },
)

strands_agent = Agent(
    model=strands_model,
    system_prompt=(
        "You are a network operations analyst for AnyComp Telecom. "
        "Use the analyze_network_report tool to extract data from PDF reports. "
        "After extraction, provide: (1) an executive summary, (2) the top issues with severity, "
        "(3) which teams should be notified, and (4) recommended immediate actions."
    ),
    tools=[analyze_network_report],
)

print("=" * 60)
print("STRANDS AGENTS — Multimodal Document Analysis")
print("=" * 60)
result = strands_agent(f'Analyze the network performance report at {REPORT_PATH}')
print(result)

## The Extraction Output

Let’s look at the raw extracted data to see what the schema produced:

In [ ]:
# Run extraction directly to inspect the structured output
extracted = extract_from_pdf(str(REPORT_PATH), NETWORK_REPORT_SCHEMA)
print(json.dumps(extracted, indent=2))

## Key Takeaway

The multimodal pattern turns unstructured documents into structured, actionable data. The JSON schema is the contract — it tells the model exactly what to extract, and you get a typed output you can validate, store, and route downstream.

This transforms a manual monthly review process into an automated pipeline: **document in → structured extraction → issue identification → team routing → action items**.